# Amazon-style Aspect Sentiment Highlighter: ModernBERT-base + R-Drop

Trains BERT, RoBERTa, DeBERTa-v3 and ModernBERT-base + R-Drop on one SemEval-2014 split (laptops + restaurants) on a Colab T4, then compares strict E2E-ABSA F1, inference throughput, and aspect recall on reviews longer than 512 tokens.

In [ ]:
# @title 1. Environment & Dependencies
import importlib.util, subprocess, sys

REQUIRED = {"seqeval": "seqeval", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "pandas": "pandas", "sentencepiece": "sentencepiece"}
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import gc, math, random, re, shutil, tarfile, time
from collections import Counter, defaultdict
from pathlib import Path

import datasets, numpy as np, pandas as pd, torch, torch.nn.functional as F, transformers
from transformers import set_seed

assert torch.cuda.is_available(), "Switch the Colab runtime to a T4 GPU"
SEED = 42
set_seed(SEED)
ART = Path("/content/artifacts")
ART.mkdir(parents=True, exist_ok=True)
MODERNBERT = "answerdotai/ModernBERT-base"

print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB)")
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__} | datasets {datasets.__version__}")
print(f"installed: {missing or 'nothing, all present'}")

In [ ]:
# @title 2. Load & Align SemEval-2014 (Laptops + Restaurants)
from datasets import load_dataset

LABELS = ["O", "B-POS", "I-POS", "B-NEG", "I-NEG", "B-NEU", "I-NEU"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}
POLARITY_MAP = {"positive": "POS", "negative": "NEG", "neutral": "NEU"}
SOURCES = {"laptops": "tomaarsen/setfit-absa-semeval-laptops", "restaurants": "tomaarsen/setfit-absa-semeval-restaurants"}


def trim(text, s, e):
    while s < e and text[s].isspace():
        s += 1
    while e > s and text[e - 1].isspace():
        e -= 1
    return s, e


def find_nth(text, span, n):
    for hay, needle in ((text, span), (text.lower(), span.lower())):
        i = -1
        for _ in range(n + 1):
            i = hay.find(needle, i + 1)
            if i < 0:
                break
        if i >= 0:
            return i
    return -1


grouped, dropped, unaligned = defaultdict(set), Counter(), 0
for domain, repo in SOURCES.items():
    for r in load_dataset(repo, split="train"):
        if r["label"] not in POLARITY_MAP:
            dropped[r["label"]] += 1
            continue
        start = find_nth(r["text"], r["span"], r["ordinal"])
        if start < 0:
            unaligned += 1
            continue
        grouped[(domain, r["text"])].add((*trim(r["text"], start, start + len(r["span"])), POLARITY_MAP[r["label"]]))

examples, seen, overlaps = [], set(), 0
for (domain, text), spanset in grouped.items():
    if text in seen:
        continue
    seen.add(text)
    spans, last_end = [], -1
    for s, e, p in sorted(spanset):
        if s < last_end:
            overlaps += 1
            continue
        spans.append((s, e, p))
        last_end = e
    examples.append({"text": text, "domain": domain, "spans": spans})

rng = random.Random(SEED)
train_examples, val_examples, test_examples = [], [], []
for domain in SOURCES:
    xs = [x for x in examples if x["domain"] == domain]
    rng.shuffle(xs)
    a, b = int(0.8 * len(xs)), int(0.9 * len(xs))
    train_examples += xs[:a]
    val_examples += xs[a:b]
    test_examples += xs[b:]

splits = {"train": train_examples, "val": val_examples, "test": test_examples}
assert not set.intersection(*({x["text"] for x in xs} for xs in splits.values())), "sentence leaked across splits"
print(f"dropped labels: {dict(dropped)} | unaligned spans: {unaligned} | overlapping spans skipped: {overlaps}")
print(pd.DataFrame([
    {"split": name, "domain": d, "sentences": sum(x["domain"] == d for x in xs),
     **Counter(p for x in xs if x["domain"] == d for _, _, p in x["spans"])}
    for name, xs in splits.items() for d in SOURCES
]).to_string(index=False))

In [ ]:
# @title 3. Tokenization, BIO Alignment & Metrics
from datasets import Dataset
from seqeval.metrics import f1_score, precision_score, recall_score
from transformers import AutoTokenizer


def tag_tokens(offsets, spans):
    tags, prev = [], None
    for s, e in offsets:
        if s == e:
            tags.append(-100)
            continue
        hit = next((sp for sp in spans if s < sp[1] and e > sp[0]), None)
        tags.append(LABEL2ID["O"] if hit is None else LABEL2ID[("I-" if hit == prev else "B-") + hit[2]])
        prev = hit
    return tags


def encode_examples(examples, tokenizer, max_length=256):
    enc = tokenizer([x["text"] for x in examples], truncation=True, max_length=max_length, return_offsets_mapping=True)
    enc["labels"] = [tag_tokens(o, x["spans"]) for o, x in zip(enc.pop("offset_mapping"), examples)]
    return Dataset.from_dict(dict(enc))


def decode_spans(text, offsets, tag_ids, scores=None):
    spans, cur = [], None
    for i, ((s, e), t) in enumerate(zip(offsets, tag_ids)):
        if s == e or t < 0:
            continue
        tag = ID2LABEL[t]
        if tag == "O":
            cur = None
            continue
        bio, pol = tag.split("-")
        if bio == "B" or cur is None or cur[2] != pol:
            cur = [s, e, pol, []]
            spans.append(cur)
        cur[1] = e
        cur[3].append(1.0 if scores is None else scores[i])
    return [(*trim(text, s, e), pol, float(np.mean(sc))) for s, e, pol, sc in spans]


@torch.inference_mode()
def predict_spans(model, tokenizer, texts, max_length, batch_size=16):
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i + batch_size]
        enc = tokenizer(chunk, truncation=True, max_length=max_length, padding=True, return_offsets_mapping=True, return_tensors="pt")
        offsets = enc.pop("offset_mapping").tolist()
        with torch.autocast("cuda", dtype=torch.float16):
            probs = model(**enc.to(model.device)).logits.float().softmax(-1)
        conf, ids = probs.max(-1)
        out += [decode_spans(t, o, i_.tolist(), c.tolist()) for t, o, i_, c in zip(chunk, offsets, ids, conf)]
    return out


def to_tags(pred_ids, label_ids):
    return [[ID2LABEL[int(p)] for p, l in zip(ps, ls) if l != -100] for ps, ls in zip(pred_ids, label_ids)]


def to_ate(seqs):
    return [[re.sub(r"-(POS|NEG|NEU)$", "-ASP", t) for t in s] for s in seqs]


def compute_metrics(eval_pred):
    logits = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
    y_true, y_pred = to_tags(eval_pred.label_ids, eval_pred.label_ids), to_tags(logits.argmax(-1), eval_pred.label_ids)
    return {"e2e_f1": f1_score(y_true, y_pred), "precision": precision_score(y_true, y_pred),
            "recall": recall_score(y_true, y_pred), "ate_f1": f1_score(to_ate(y_true), to_ate(y_pred))}


mb_tok = AutoTokenizer.from_pretrained(MODERNBERT)
enc = mb_tok([x["text"] for x in examples], truncation=True, max_length=256, return_offsets_mapping=True)
gold = recovered = 0
for x, offsets in zip(examples, enc["offset_mapping"]):
    decoded = {s[:3] for s in decode_spans(x["text"], offsets, tag_tokens(offsets, x["spans"]))}
    gold += len(x["spans"])
    recovered += len(decoded & set(x["spans"]))
print(f"round-trip (ModernBERT offsets -> BIO -> char spans): {recovered}/{gold} = {recovered / gold:.2%} spans recovered exactly")
assert recovered / gold > 0.99, "BIO alignment loses spans; check tokenizer offsets"

In [ ]:
# @title 4. R-Drop Trainer (Custom Loss for Sequence Tagging)
from transformers import Trainer


class RDropTokenTrainer(Trainer):
    def __init__(self, *args, rdrop_alpha=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.rdrop_alpha = rdrop_alpha

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        feats = {k: v for k, v in inputs.items() if k != "labels"}

        def ce(logits):
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)

        out = model(**feats)
        loss = ce(out.logits)
        if model.training and self.rdrop_alpha > 0:
            out2 = model(**feats)
            mask = labels != -100
            p = F.log_softmax(out.logits[mask].float(), -1)
            q = F.log_softmax(out2.logits[mask].float(), -1)
            kl = 0.5 * (F.kl_div(p, q, log_target=True, reduction="batchmean") + F.kl_div(q, p, log_target=True, reduction="batchmean"))
            loss = 0.5 * (loss + ce(out2.logits)) + self.rdrop_alpha * kl
        return (loss, out) if return_outputs else loss

In [ ]:
# @title 5. Train & Benchmark 4 Models on Identical Split
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments

OURS = "ModernBERT-base + R-Drop (Ours)"
MODELS = [
    dict(name="BERT-base", ckpt="bert-base-uncased", rdrop=0.0, lr=3e-5, epochs=5, fp16=True, max_len=512),
    dict(name="RoBERTa-base", ckpt="roberta-base", rdrop=0.0, lr=3e-5, epochs=5, fp16=True, max_len=512),
    dict(name="DeBERTa-v3-base", ckpt="microsoft/deberta-v3-base", rdrop=0.0, lr=3e-5, epochs=5, fp16=False, max_len=512),  # fp16 training overflows on T4
    dict(name=OURS, ckpt=MODERNBERT, rdrop=1.5, lr=5e-5, epochs=6, fp16=True, max_len=1024),
]
BATCH_SIZE, WEIGHT_DECAY, WARMUP_RATIO = 16, 0.01, 0.1


def build_long_reviews(n=30, head_tokens=600, max_tokens=900):
    rng, reviews = random.Random(SEED), []
    ntok = lambda t: len(mb_tok(t)["input_ids"])
    for _ in range(n):
        pool = iter(rng.sample(test_examples, len(test_examples)))
        text, gold = "", []
        while ntok(text) < head_tokens:
            text += next(pool)["text"] + "\n\n"
        for x in pool:
            if ntok(text + x["text"]) > max_tokens:
                break
            gold += [(s + len(text), e + len(text), p) for s, e, p in x["spans"]]
            text += x["text"] + "\n\n"
        reviews.append({"text": text.rstrip(), "spans": gold})
    return reviews


@torch.inference_mode()
def benchmark_speed(model, tokenizer, texts, batch_size=32, runs=5):
    batches = [tokenizer(texts[i:i + batch_size], truncation=True, max_length=256, padding=True, return_tensors="pt").to(model.device)
               for i in range(0, len(texts), batch_size)]
    torch.cuda.reset_peak_memory_stats()

    def one_run():
        lat, nonfinite = [], 0
        for b in batches:
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**b).logits
            logits.argmax(-1).cpu()
            lat.append(time.perf_counter() - t0)
            nonfinite += (~torch.isfinite(logits)).sum().item()
        return lat, nonfinite

    one_run()
    measured = [one_run() for _ in range(runs)]
    tput = [len(texts) / sum(lat) for lat, _ in measured]
    return {"throughput": float(np.median(tput)), "tput_min": min(tput), "tput_max": max(tput),
            "p95_ms": float(np.percentile(np.concatenate([lat for lat, _ in measured]) * 1000, 95)),
            "peak_mem_mb": torch.cuda.max_memory_allocated() / 2**20, "nonfinite_logits": sum(n for _, n in measured)}


long_reviews = build_long_reviews()
long_gold = sum(len(r["spans"]) for r in long_reviews)
long_lens = [len(mb_tok(r["text"])["input_ids"]) for r in long_reviews]
print(f"long reviews: {len(long_reviews)} | ModernBERT tokens {min(long_lens)}-{max(long_lens)} | {long_gold} gold aspects, all placed after token 600")
speed_texts = [x["text"] for x in random.Random(SEED).sample(test_examples, min(256, len(test_examples)))]

results = []
for cfg in MODELS:
    print(f"\n=== {cfg['name']} ({cfg['ckpt']}) ===")
    tok = AutoTokenizer.from_pretrained(cfg["ckpt"])
    train_ds, val_ds, test_ds = (encode_examples(xs, tok) for xs in (train_examples, val_examples, test_examples))
    steps = math.ceil(len(train_ds) / BATCH_SIZE) * cfg["epochs"]
    t0 = time.perf_counter()
    for fp16 in dict.fromkeys([cfg["fp16"], False]):
        set_seed(SEED)
        model = AutoModelForTokenClassification.from_pretrained(cfg["ckpt"], num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)
        args = TrainingArguments(
            output_dir=f"/content/runs/{cfg['ckpt'].split('/')[-1]}", learning_rate=cfg["lr"], num_train_epochs=cfg["epochs"],
            per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=64, weight_decay=WEIGHT_DECAY,
            warmup_steps=int(WARMUP_RATIO * steps), max_grad_norm=1.0, fp16=fp16,
            eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True,
            metric_for_best_model="e2e_f1", greater_is_better=True, logging_steps=50, logging_nan_inf_filter=False,
            report_to="none", seed=SEED,
        )
        trainer = RDropTokenTrainer(
            model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, processing_class=tok,
            data_collator=DataCollatorForTokenClassification(tok, pad_to_multiple_of=8), compute_metrics=compute_metrics,
            rdrop_alpha=cfg["rdrop"],
        )
        trainer.train()
        if all(math.isfinite(h["loss"]) for h in trainer.state.log_history if "loss" in h):
            break
        print(f"non-finite training loss with fp16=True; retraining {cfg['name']} in fp32")
    train_min = (time.perf_counter() - t0) / 60
    model = trainer.model.eval()

    test = trainer.predict(test_ds).metrics
    speed = benchmark_speed(model, tok, speed_texts)
    preds = predict_spans(model, tok, [r["text"] for r in long_reviews], cfg["max_len"])
    long_hits = sum(len({s[:3] for s in p} & set(r["spans"])) for p, r in zip(preds, long_reviews))
    results.append({
        "model": cfg["name"], "params_M": sum(p.numel() for p in model.parameters()) / 1e6, "fp16_train": fp16,
        "train_min": train_min, "best_val_e2e_f1": trainer.state.best_metric,
        "e2e_f1": test["test_e2e_f1"], "ate_f1": test["test_ate_f1"], "precision": test["test_precision"], "recall": test["test_recall"],
        **speed, "long_max_len": cfg["max_len"], "long_recall": long_hits / long_gold,
    })
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in results[-1].items()})

    if cfg["name"] == OURS:
        trainer.save_model(ART / "modernbert-absa")
        with tarfile.open(ART / "modernbert-absa.tar.gz", "w:gz") as tar:
            tar.add(ART / "modernbert-absa", arcname="modernbert-absa")
        print(f"saved {ART / 'modernbert-absa'} and {ART / 'modernbert-absa.tar.gz'} ({(ART / 'modernbert-absa.tar.gz').stat().st_size / 2**20:.0f} MB)")

    shutil.rmtree(args.output_dir, ignore_errors=True)
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# @title 6. Benchmark Comparison Table & Pareto Plot
import matplotlib.pyplot as plt

df = pd.DataFrame(results)
df.to_csv(ART / "benchmark_results.csv", index=False)
with pd.option_context("display.width", 250, "display.max_columns", None, "display.float_format", "{:.3f}".format):
    print(df.to_string(index=False))

ours = df[df.model == OURS].iloc[0]
print(f"\nThroughput: median of 5 runs after 1 warmup, {len(speed_texts)} test sentences, batch 32, fp16 autocast, T4 SDPA.")
print("Single training seed per model: E2E-F1 gaps under ~1 point are within seed-to-seed noise.")
for _, r in df[df.model != OURS].iterrows():
    print(f"Ours vs {r.model:16s} E2E-F1 {100 * (ours.e2e_f1 - r.e2e_f1):+.2f} pts | speed {ours.throughput / r.throughput:.2f}x "
          f"| long-review recall {ours.long_recall:.1%} vs {r.long_recall:.1%}")

colors = ["#d62728" if m == OURS else "#1f77b4" for m in df.model]
fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(14, 5.5))
ax_a.errorbar(df.throughput, df.e2e_f1, xerr=[df.throughput - df.tput_min, df.tput_max - df.throughput], fmt="none", ecolor="gray", capsize=4)
ax_a.scatter(df.throughput, df.e2e_f1, c=colors, s=140, zorder=3)
for _, r in df.iterrows():
    ax_a.annotate(r.model, (r.throughput, r.e2e_f1), textcoords="offset points", xytext=(8, 6), fontsize=9)
frontier = df.sort_values("throughput", ascending=False)
frontier = frontier[frontier.e2e_f1 >= frontier.e2e_f1.cummax()]
ax_a.plot(frontier.throughput, frontier.e2e_f1, "--", color="#d62728", alpha=0.5, label="Pareto frontier")
ax_a.set(xlabel="Inference throughput (sentences/s, T4 fp16)", ylabel="Test E2E-ABSA micro-F1 (strict)", title="A. Accuracy vs speed")
ax_a.legend(loc="lower right")
ax_a.grid(alpha=0.3)

bars = ax_b.bar(df.model, df.long_recall, color=colors)
ax_b.bar_label(bars, labels=[f"{v:.0%}\n(max_len {m})" for v, m in zip(df.long_recall, df.long_max_len)], fontsize=9)
ax_b.set(ylabel="Strict aspect recall", ylim=(0, 1.15), title=f"B. Aspects past token 512 ({len(long_reviews)} reviews, {long_gold} aspects)")
ax_b.tick_params(axis="x", labelrotation=15)
ax_b.grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(ART / "pareto_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# @title 7. Live Amazon Review Demo (Zero-Shot Transfer)
REVIEWS = [
    "Amazing noise cancellation, but the ear cushions get hot after an hour.",
    "The battery life is incredible and lasts a full week, though the charging case feels cheap.",
    "Sound quality is crisp and the bass is punchy. The touch controls are way too sensitive.",
    "Bluetooth pairing was instant, but the microphone makes my voice sound muffled on calls.",
    "These headphones are comfortable for long flights and the carrying case is sturdy.",
    "The app is buggy and keeps crashing, yet the equalizer presets sound great.",
    "This laptop's screen is gorgeous and bright, but the keyboard flexes when I type.",
    "Performance is snappy for coding and the trackpad is smooth. The fan gets loud under load.",
    "Battery barely lasts four hours and the charger is bulky. At least the build quality feels premium.",
    "Customer support was helpful and shipping was fast, but the webcam is grainy.",
    "The speakers are tinny and the price is too high for what you get.",
    "Windows came preinstalled with bloatware, though the SSD boots in seconds and the hinge is solid.",
]

demo_tok = AutoTokenizer.from_pretrained(ART / "modernbert-absa")
demo_model = AutoModelForTokenClassification.from_pretrained(ART / "modernbert-absa").cuda().eval()
SYMBOL = {"POS": "+", "NEG": "-", "NEU": "~"}
summary = defaultdict(Counter)
for i, (text, spans) in enumerate(zip(REVIEWS, predict_spans(demo_model, demo_tok, REVIEWS, max_length=1024)), 1):
    print(f"[{i:2d}] {text}")
    for s, e, pol, score in spans:
        print(f"      {SYMBOL[pol]} {text[s:e]!r:28s} {pol}  {score:.2f}")
        summary[text[s:e].lower()][pol] += 1

print("\nAspect summary  (aspect | +pos / -neg / ~neu)")
for aspect, c in sorted(summary.items(), key=lambda kv: -sum(kv[1].values())):
    print(f"  [ {aspect} | +{c['POS']} / -{c['NEG']} / ~{c['NEU']} ]")